## Task 1: Sentence Transformer Implementation

### Step 1: Import Required Libraries

In [8]:
import torch # PyTorch is used to handle tensors and models.
from transformers import BertTokenizer, BertModel # Provides pretrained BERT models and tokenizers.
import numpy as np # For converting tensor outputs to a format easier to print and inspect.
import torch.nn as nn  # Provides base classes for building neural network layers like Linear, Conv, etc.
import torch.nn.functional as F  # Contains functional APIs for layers and operations like activation functions (ReLU, softmax), loss functions, etc.
from torch.utils.data import DataLoader, TensorDataset  # Helps in batching and loading data efficiently for training and evaluation.
from sklearn.metrics import accuracy_score  # Used to compute accuracy as a performance metric for classification tasks.

### I am leveraging the pre-trained bert-base-uncased model.
### Step 2: Define the SentenceTransformerModel Class
###  Step 3: Implement Mean Pooling
###  Step 4: Encode Sentences

In [2]:
class SentenceTransformerModel(torch.nn.Module): # defining a custom PyTorch model by subclassing torch.nn.Module.
    def __init__(self, model_name='bert-base-uncased'): # Using pretrained "bert-base-uncased" as the default model which is loaded whenever the object is created
        super(SentenceTransformerModel, self).__init__() # initializing the parent class (torch.nn.Module)
        self.bert = BertModel.from_pretrained(model_name) # # Loading the BERT transformer model with pretrained weights from the specified model name.
        self.tokenizer = BertTokenizer.from_pretrained(model_name) # Loads the corresponding tokenizer for the specified BERT model.
    
    def mean_pooling(self, model_output, attention_mask):
        # Extract the token embeddings from the model output (shape: [batch_size, seq_len, hidden_size])
        token_embeddings = model_output.last_hidden_state
    
        # Expand the attention mask to match the dimensions of token_embeddings
        # This ensures that padding tokens are excluded in the pooling step
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size())
    
        # Perform mean pooling: sum the embeddings of valid tokens and divide by the number of valid (non-padded) tokens
        # This results in a fixed-size embedding per sentence
        return (token_embeddings * input_mask_expanded).sum(1) / input_mask_expanded.sum(1)


    def encode(self, sentences):
        encoded_input = self.tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')
    
        # Disable gradient computation since we're only encoding (not training)
        with torch.no_grad():
            # Pass the tokenized input through the BERT model to get token embeddings
            model_output = self.bert(**encoded_input)
    
        # Apply mean pooling to get a fixed-size sentence embedding from token-level outputs
        sentence_embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])
    
        # Return the final sentence embeddings (one per input sentence)
        return sentence_embeddings

### Step 5: Initialize and Use the Model

In [3]:
model = SentenceTransformerModel()

sentences = [
    "This is Sai Harsha Vardhan: A Machine Learning specialist",
    "A quick brown fox jumps over the lazy dog.",
    "Machine learning is fascinating.",
    "The weather today is sunny and bright."
]


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

### Step 6: Encode and Display Embeddings

In [4]:
# Encode the list of sentences to obtain fixed-size embeddings using the model
embeddings = model.encode(sentences)
# Detach the embeddings from the computation graph and convert them to a NumPy array
embeddings_np = embeddings.detach().numpy()
# Iterate through each sentence embedding
for i, emb in enumerate(embeddings_np):
    # Print the first 10 dimensions of the 768-dimensional embedding for each sentence
    print(f"Sentence {i+1} embedding (dim={len(emb)}):\n{emb[:10]}...")


Sentence 1 embedding (dim=768):
[ 0.09599355 -0.00144955  0.03809299 -0.21845683  0.3177811  -0.5443898
  0.34633553  0.30962166  0.06223265  0.15145913]...
Sentence 2 embedding (dim=768):
[ 0.18255323  0.09675284  0.01449239 -0.00548042  0.37569115 -0.07019629
  0.06506577  0.4533459   0.06035462 -0.08866962]...
Sentence 3 embedding (dim=768):
[ 0.15958557  0.07248507 -0.14404048  0.04608745  0.4271044  -0.51639056
 -0.04411336  0.58455926  0.00517792 -0.51480407]...
Sentence 4 embedding (dim=768):
[-0.09487744 -0.27084324 -0.03123956  0.03418256  0.29597646 -0.47443858
 -0.12302394  1.1176074   0.00590292 -0.6479481 ]...


## Task 2: Multi-Task Learning Expansion 

### Task A: Sentence Classification
### Retain the sentence embedding logic from the earlier model.

### Add a classification head (a Linear layer).

### Ensure the model can be trained with labeled classification data.

### Support both encoding and prediction via clearly defined methods.

### Key Changes:
#### self.classifier........Added a linear layer to classify sentence embeddings into task-specific classes
#### forward().........Modified to return both sentence embeddings and tasks output
#### Output Format..........Returns a dictionary for multi-task expansion readiness

In [5]:
class MultiTaskSentenceTransformer(nn.Module):
    def __init__(self, model_name='bert-base-uncased', num_classes=3):
        super(MultiTaskSentenceTransformer, self).__init__()

        # Load the pre-trained BERT model and tokenizer
        self.bert = BertModel.from_pretrained(model_name)
        self.tokenizer = BertTokenizer.from_pretrained(model_name)

        # Define a sentence classification head (for Task A)
        # It maps 768-dimensional BERT embeddings to num_classes outputs
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def mean_pooling(self, model_output, attention_mask):
        # Extract the token embeddings
        token_embeddings = model_output.last_hidden_state

        # Expand the attention mask to match the embedding dimensions
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size())

        # Perform mean pooling by averaging only non-padding token embeddings
        return (token_embeddings * input_mask_expanded).sum(1) / input_mask_expanded.sum(1)

    def forward(self, sentences):
        # Tokenize input sentences for the BERT model
        encoded_input = self.tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')

        # Disable gradient tracking if only inference (optional)
        with torch.no_grad():
            model_output = self.bert(**encoded_input)

        # Get fixed-length sentence embeddings using mean pooling
        sentence_embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])

        # --- Task A: Sentence Classification ---
        # Pass the sentence embeddings through the classifier to get logits
        class_logits = self.classifier(sentence_embeddings)

        # Return both: embeddings and classification output (multi-task learning ready)
        return {
            "embeddings": sentence_embeddings,
            "classification_logits": class_logits
        }

### Example Usage

In [6]:
# Instantiate model
model = MultiTaskSentenceTransformer(num_classes=3)  # Suppose we have 3 classes

# Define sample sentences
sentences = [
    "The movie was fantastic and exciting.",
    "I hated the food and service.",
    "The product arrived on time and was well-packaged."
]

# Get predictions
outputs = model(sentences)

# Extract classification logits and embeddings
logits = outputs["classification_logits"]
embeddings = outputs["embeddings"]

# Convert logits to class predictions
predicted_classes = torch.argmax(logits, dim=1)

# Print results
for i, sent in enumerate(sentences):
    print(f"Sentence: {sent}")
    print(f"Predicted class: {predicted_classes[i].item()}")
    print(f"Embedding (first 5 dims): {embeddings[i][:5].detach().numpy()}\n")


Sentence: The movie was fantastic and exciting.
Predicted class: 2
Embedding (first 5 dims): [ 0.3156766  -0.25783688  0.02960825  0.39757702  0.195079  ]

Sentence: I hated the food and service.
Predicted class: 2
Embedding (first 5 dims): [ 0.2883285  -0.00120383 -0.2196141   0.2172923  -0.03111824]

Sentence: The product arrived on time and was well-packaged.
Predicted class: 2
Embedding (first 5 dims): [-0.18662435 -0.30690414  0.22416748  0.436578   -0.1101101 ]



### Task B: Named Entity Recognition (NER)
### Key Changes:
#### Output Type......Label per token (sequence labeling)
#### Pooling......No need because we need predictions per token
#### Output Layer.......nn.Linear(hidden_size, num_ner_labels)
#### BERT Output Used.......Token-level embeddings (last_hidden_state)

In [7]:
class SentenceTransformerForNER(nn.Module):
    def __init__(self, model_name='bert-base-uncased', num_ner_labels=5):
        super(SentenceTransformerForNER, self).__init__()

        # Load the pre-trained BERT model and tokenizer
        self.bert = BertModel.from_pretrained(model_name)
        self.tokenizer = BertTokenizer.from_pretrained(model_name)

        # Task B: Token-level NER head
        # Each token's hidden state is mapped to one of the NER labels
        self.ner_head = nn.Linear(self.bert.config.hidden_size, num_ner_labels)

    def forward(self, sentences):
        # Tokenize the input sentences with padding/truncation and return PyTorch tensors
        encoded_input = self.tokenizer(sentences, padding=True, truncation=True, return_tensors='pt', return_attention_mask=True)

        # Pass input through BERT to get token-level hidden states
        model_output = self.bert(**encoded_input)

        # model_output.last_hidden_state shape: [batch_size, seq_len, hidden_size]
        token_embeddings = model_output.last_hidden_state

        # Pass token embeddings through the NER classification head
        # Output shape: [batch_size, seq_len, num_ner_labels]
        ner_logits = self.ner_head(token_embeddings)

        return {
            "ner_logits": ner_logits,
            "input_ids": encoded_input['input_ids'],
            "attention_mask": encoded_input['attention_mask']
        }

In [8]:
# Instantiate model with 5 dummy NER classes
model = SentenceTransformerForNER(num_ner_labels=5)

# Sample sentence
sentence = ["Steve Jobs founded Apple in California."]

# Get NER logits from the model
outputs = model(sentence)
logits = outputs["ner_logits"]

# Predict token labels using argmax
predicted_ner_labels = torch.argmax(logits, dim=-1)

# Convert token IDs back to tokens
tokens = model.tokenizer.convert_ids_to_tokens(outputs["input_ids"][0])

# Display results
print("NER Predictions:")
for token, label_id in zip(tokens, predicted_ner_labels[0]):
    print(f"{token:15} -> Label {label_id.item()}")

NER Predictions:
[CLS]           -> Label 0
steve           -> Label 3
jobs            -> Label 3
founded         -> Label 0
apple           -> Label 3
in              -> Label 0
california      -> Label 4
.               -> Label 0
[SEP]           -> Label 0


### Task 3: Training Considerations 

#### 1. Entire Network is Frozen
##### What it Means:
 You freeze all layers, including the BERT backbone and task-specific heads.
 No part of the model gets updated during training.

##### Advantages:
 Fastest inference – great for zero-shot or fixed feature extraction.
 No risk of overfitting – ideal if you have very limited labeled data.
 Useful for prototyping or when using BERT embeddings purely as features for downstream models.

##### Implications:
 You cannot learn new tasks or adapt to domain-specific data.
 Both Task A (classification) and Task B (NER) rely on pretrained knowledge only.

##### Use When:
 You're using BERT as a static feature extractor.
 You're evaluating performance of task-specific heads as-is, without training.

#### 2. Only Transformer Backbone is Frozen
##### What it Means:
 Freeze the BERT layers (self.bert).
 Allow only task-specific heads (e.g., classifier or NER head) to learn and update.

##### Advantages:
 Faster training than fine-tuning the whole model.
 Retains general language knowledge from BERT.
 Reduces risk of catastrophic forgetting.
 Requires less compute and fewer epochs.

##### Implications:
 The heads will try to map general-purpose embeddings to your tasks, which may work well if:
 Your task is simple Or similar to the tasks BERT was pretrained on
 You may not get optimal results for domain-specific vocabulary or structure.

##### Use When:
 You want fast training with decent performance.
 You’re working with moderate-sized datasets.
 You want to re-use BERT and just train task adapters.

#### 3. Only One Task-Specific Head is Frozen
##### What it Means:
 You freeze either the classifier (Task A) or the NER head (Task B).
 BERT and the other head remain trainable.

##### Advantages:
 Enables focused fine-tuning for just one task.
 Preserves knowledge in the frozen head (e.g., already trained NER or classifier).
 Useful when you're adding a new task without disturbing an existing one.

##### Implications:
 The frozen task head won't adapt to new data, which might hurt performance if:
 The domain has shifted Or new data is richer/different than training data.

##### Use When:
 You're performing sequential task training (multi-task to continual learning).
 You want to preserve performance on an existing task while training another.

### Lets consider the following scenario for better understanding:
#### Transfer Learning for Named Entity Recognition (NER) in Medical Texts
##### 1. Choice of Pre-trained Model
I would use a domain-specific model such as BioBERT or SciBERT, which are fine-tuned on biomedical corpora like PubMed. These models better understand medical language and are ideal for tasks like extracting diseases, medications, or symptoms.

If no domain-specific model is available, a general-purpose model like bert-base-cased can be used, especially when case sensitivity matters.

##### 2. Layers to Freeze/Unfreeze
Freeze lower transformer layers (e.g., layers 0–6): These capture foundational language understanding such as grammar, syntax, and general semantics.

Unfreeze upper transformer layers (e.g., layers 7–11): These can adapt to domain-specific patterns and contextual nuances found in medical texts.

Unfreeze and train the task-specific NER head: Since our custom entity labels (like B-DISEASE, B-MEDICATION) are new, the classification head must be trained from scratch.

##### 3. Rationale
Freezing lower layers helps retain BERT's general language knowledge and reduces training time and the risk of overfitting—especially useful when working with small labeled datasets.

Unfreezing upper layers allows the model to learn domain-specific relationships that are critical for identifying entities in specialized fields like healthcare.

Training the NER head is necessary because the pre-trained model doesn’t know your task-specific labels—it learns them during fine-tuning.

### Task 4: Training Loop Implementation

#### Assumptions:
I am using hypothetical datasets for both tasks (random tensors for now).

Task A is sentence-level, using CrossEntropyLoss.

Task B is token-level, using CrossEntropyLoss with attention masking.

I am optimizing both tasks jointly, using a combined loss.

Metrics: I am just printing simple accuracy for both tasks (to show the concept)

#### Multi-task Learning Behavior
##### In each batch:

We get inputs and labels for both tasks.

We forward through the shared BERT model.

Compute loss for each task.

Combine losses with equal weights.

Backpropagate and update.

In [2]:
# Simulated model from Task 2
class MultiTaskSentenceTransformer(nn.Module):
    def __init__(self, model_name='bert-base-uncased', num_classes=3, num_ner_labels=5):
        super(MultiTaskSentenceTransformer, self).__init__()
        from transformers import BertModel
        self.bert = BertModel.from_pretrained(model_name)
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)  # Task A
        self.ner_head = nn.Linear(self.bert.config.hidden_size, num_ner_labels)  # Task B

    def mean_pooling(self, output, mask):
        token_embeddings = output.last_hidden_state
        input_mask_expanded = mask.unsqueeze(-1).expand(token_embeddings.size())
        return (token_embeddings * input_mask_expanded).sum(1) / input_mask_expanded.sum(1)

    def forward(self, input_ids, attention_mask):
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sentence_embeddings = self.mean_pooling(output, attention_mask)
        sentence_logits = self.classifier(sentence_embeddings)               # Task A
        token_logits = self.ner_head(output.last_hidden_state)              # Task B
        return sentence_logits, token_logits

In [3]:
# Hyperparameters
num_classes = 3
num_ner_labels = 5
batch_size = 2
seq_len = 10
vocab_size = 30522  # typical BERT vocab

In [4]:
# Simulated input and target data
X_input_ids = torch.randint(0, vocab_size, (20, seq_len))       # Simulated token IDs
attention_masks = torch.ones_like(X_input_ids)                  # All tokens are valid (no padding here)
y_sent = torch.randint(0, num_classes, (20,))                   # One label per sentence (Task A)
y_ner = torch.randint(0, num_ner_labels, (20, seq_len))         # One label per token (Task B)

In [5]:
# Wrap in a TensorDataset and DataLoader
dataset = TensorDataset(X_input_ids, attention_masks, y_sent, y_ner)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [6]:
# ======== INITIALIZE MODEL, LOSSES, OPTIMIZER ==========

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MultiTaskSentenceTransformer(num_classes=num_classes, num_ner_labels=num_ner_labels).to(device)

criterion_sent = nn.CrossEntropyLoss()         # For sentence classification (Task A)
criterion_ner = nn.CrossEntropyLoss()          # For token classification (Task B)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [7]:
# ======== TRAINING LOOP STARTS HERE ==========
model.train()

for epoch in range(3):  # Simulated 3 epochs
    print(f"\nEpoch {epoch+1}")
    total_sent_loss = 0
    total_ner_loss = 0

    for batch in loader:
        input_ids, mask, labels_sent, labels_ner = [x.to(device) for x in batch]

        # ---- Forward Pass for Both Tasks ----
        logits_sent, logits_ner = model(input_ids, mask)

        # ---- Compute Losses ----
        loss_sent = criterion_sent(logits_sent, labels_sent)

        # Reshape NER logits and labels for CrossEntropyLoss
        logits_ner_flat = logits_ner.view(-1, num_ner_labels)
        labels_ner_flat = labels_ner.view(-1)
        loss_ner = criterion_ner(logits_ner_flat, labels_ner_flat)

        # ---- Total loss (equal weight for both tasks) ----
        loss = loss_sent + loss_ner

        # ---- Backward Pass & Optimizer Step ----
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_sent_loss += loss_sent.item()
        total_ner_loss += loss_ner.item()

    print(f"Sentence Loss: {total_sent_loss:.4f} | NER Loss: {total_ner_loss:.4f}")

    # ========METRICS ON LAST BATCH ==========
    # Accuracy for Task A
    preds_sent = torch.argmax(logits_sent, dim=1)
    acc_sent = accuracy_score(labels_sent.cpu(), preds_sent.cpu())

    # Token Accuracy for Task B (just to show logic)
    preds_ner = torch.argmax(logits_ner, dim=2)
    correct_tokens = (preds_ner == labels_ner).float()
    acc_ner = correct_tokens.sum() / correct_tokens.numel()

    print(f"Task A (Sentence Classification) Accuracy: {acc_sent:.2f}")
    print(f"Task B (NER) Token Accuracy: {acc_ner:.2f}")


Epoch 1
Sentence Loss: 12.0850 | NER Loss: 16.3606
Task A (Sentence Classification) Accuracy: 0.00
Task B (NER) Token Accuracy: 0.25

Epoch 2
Sentence Loss: 9.9144 | NER Loss: 16.0622
Task A (Sentence Classification) Accuracy: 0.50
Task B (NER) Token Accuracy: 0.15

Epoch 3
Sentence Loss: 8.2702 | NER Loss: 15.8045
Task A (Sentence Classification) Accuracy: 1.00
Task B (NER) Token Accuracy: 0.30
